## Imports

In [ ]:
import pandas as pd

population_csv = pd.read_csv('src/src_insee/DS_RP_POPULATION_COMP_2022_CSV_FR/DS_RP_POPULATION_COMP_2022_data.csv', sep=';', dtype={'GEO': str, 'PCS': str})
df_population = pd.DataFrame(population_csv)

conjugality_csv = pd.read_csv('src/src_insee/DS_RP_MENAGES_PRINC_2022_CSV_FR/DS_RP_MENAGES_PRINC_2022_data.csv', sep=';', dtype={'GEO': str, 'COUPLE': str})
df_conjugality = pd.DataFrame(conjugality_csv)

household_csv = pd.read_csv('src/src_insee/DS_RP_MENAGES_COMP_2022_CSV_FR/DS_RP_MENAGES_COMP_2022_data.csv', sep=';', dtype={'GEO': str})
df_household = pd.DataFrame(household_csv)


## Population

### Suppression des données inutiles

- Supression de la colonne RP_MEASURE : valeur unique
- GEO_OBJECT : on conserve uniquement la valeur 'COM' (commune)
- Supression de GEO_OBJECT après avoir filtré les communes

In [ ]:
# Suppression des lignes et colonnes inutiles (RP_MEASURE a une valeur unique + on veut garder uniquement les GEO_OBJECT communes)
df_population = df_population[df_population['GEO_OBJECT'] == 'COM']
df_population = df_population.drop(['GEO_OBJECT', 'RP_MEASURE'], axis=1)
df_population = df_population.sort_values(['TIME_PERIOD', 'GEO'])


### Premier aperçu des données

In [ ]:
duplicates_count = df_population.duplicated().sum()
print('Nombre de doublons :', duplicates_count)

null_values_count = df_population.isnull().sum()
print(f'\nNombre de valeurs manquantes : {null_values_count}')

distinct_cities = len(df_population['GEO'].unique())
print(f'\nNombre de communes : {distinct_cities}')

display(df_population)

### Structuration des données

- Transformer la colonne SEX en 3 colonnes : nombre d'hommes, nombre de femmes, population totale

In [ ]:
df_population_sex = df_population[
    (df_population['AGE'] == 'Y_GE15') & (df_population['PCS'] == '_T')
].pivot_table(
    index=['GEO', 'TIME_PERIOD'],
    columns='SEX',
    values='OBS_VALUE',
    aggfunc='sum',
    fill_value=0
).rename(columns={'F': 'Femmes', 'M': 'Hommes', '_T': 'Population active'})

df_population_sex = df_population_sex.reset_index()

display(df_population_sex)

- Transformer la colonne PCS en une colonne par catégorie socio-professionnelle

In [ ]:
df_population_pcs = df_population[
    (df_population['AGE'] == 'Y_GE15') & (df_population['SEX'] == '_T')
].pivot_table(
    index=['GEO', 'TIME_PERIOD'],
    columns='PCS',
    values='OBS_VALUE',
    aggfunc='sum',
    fill_value=0
).rename(columns={
    '1': 'Agriculteurs',
    '2': 'Artisans',
    '3': 'Cadres',
    '4': 'Intermédiaires',
    '5': 'Employés',
    '6': 'Ouvriers',
    '7': 'Retraités',
    '8': 'Etudiants',
    '9': 'Inactifs',
    '_T': 'Population active'
})

df_population_pcs = df_population_pcs.reset_index()

display(df_population_pcs)

### Dataframe population final

In [ ]:
# On regroupe tout en un seul dataframe en supprimant les colonnes en double

df_population_final = pd.merge(df_population_sex, df_population_pcs, on=['GEO', 'TIME_PERIOD', 'Population active'], how='inner')

display(df_population_final)

## Conjugality

### Suppression des données inutiles

- Suppression de la colonne FREQ qui comporte une seule valeur
- GEO_OBJECT : on conserve uniquement la valeur 'COM' (commune)
- Supression de GEO_OBJECT après avoir filtré les communes
- RP_MEASURE : on conserve la mesure POP, identique au dataframe Population
- Suppression de NOC et OCS qui ont des valeurs uniques après avoir filtré RP_MEASURE

In [ ]:
# Suppression des lignes et colonnes inutiles
df_conjugality = df_conjugality[(df_conjugality['GEO_OBJECT'] == 'COM') & (df_conjugality['RP_MEASURE'] == 'POP')]
df_conjugality = df_conjugality.drop(['GEO_OBJECT', 'FREQ', 'RP_MEASURE', 'NOC', 'OCS'], axis=1)
df_conjugality = df_conjugality.sort_values(['TIME_PERIOD', 'GEO'])


### Premier aperçu des données

In [ ]:
duplicates_count = df_conjugality.duplicated().sum()
print('Nombre de doublons :', duplicates_count)

null_values_count = df_conjugality.isnull().sum()
print(f'\nNombre de valeurs manquantes : {null_values_count}')

distinct_cities = len(df_conjugality['GEO'].unique())
print(f'\nNombre de communes : {distinct_cities}')

display(df_conjugality)

### Structuration des données

- Transformer la colonne AGE en une colonne par tranche d'âge

In [ ]:
df_conjugality_age = df_conjugality[
    (df_conjugality['COUPLE'] == '_T') & (df_conjugality['CIVIL_STATUS'] == '_T')
].pivot_table(
    index=['GEO', 'TIME_PERIOD'],
    columns='AGE',
    values='OBS_VALUE',
    aggfunc='sum',
    fill_value=0
).rename(columns={
    'Y15T24': '15-24 ans',
    'Y25T39': '25-39 ans',
    'Y40T54': '40-54 ans',
    'Y55T64': '55-64 ans',
    'Y65T79': '65-79 ans',
    'Y_GE80': '80 ans et +',
    'Y_GE15': 'Population active'
})


df_conjugality_age = df_conjugality_age.reset_index()

display(df_conjugality_age)

- Transformer la colonne CIVIL_STATUS en une colonne par statut marital
- Supprimer la colonne 2T6 (Non mariés) qui regroupe les veufs, les divorcés et les célibataires et n'apparaît que pour 2016 (il faudra ensuite imputer les valeurs manquantes pour cette année-là lors de la création du modèle)

In [ ]:
df_conjugality_status = df_conjugality[
    (df_conjugality['COUPLE'] == '_T') & (df_conjugality['AGE'] == 'Y_GE15')
].pivot_table(
    index=['GEO', 'TIME_PERIOD'],
    columns='CIVIL_STATUS',
    values='OBS_VALUE',
    aggfunc='sum',
    fill_value=0
).rename(columns={
    '1': 'Mariés',
    '2': 'Pacsés',
    '3': 'Concubinage',
    '4': 'Veufs',
    '5': 'Divorcés',
    '6': 'Célibataires',
    '_T': 'Population active',
}).drop('2T6', axis=1)

df_conjugality_status = df_conjugality_status.reset_index()

display(df_conjugality_status)

- Transformer la colonne COUPLE en population en couple et population célibataire

In [ ]:
df_conjugality_couple = df_conjugality[
    (df_conjugality['CIVIL_STATUS'] == '_T') & (df_conjugality['AGE'] == 'Y_GE15')
].pivot_table(
    index=['GEO', 'TIME_PERIOD'],
    columns='COUPLE',
    values='OBS_VALUE',
    aggfunc='sum',
    fill_value=0
).rename(columns={
    '0': 'Pas en couple',
    '1': 'En couple',
    '_T': 'Population active'
})

df_conjugality_couple = df_conjugality_couple.reset_index()

display(df_conjugality_couple)

### Dataframe conjugality final

In [ ]:
# On regroupe tout en un seul dataframe en supprimant les colonnes en double

df_conjugality_merged = pd.merge(df_conjugality_age, df_conjugality_status, on=['GEO', 'TIME_PERIOD', 'Population active'], how='inner')
df_conjugality_final = pd.merge(df_conjugality_merged, df_conjugality_couple, on=['GEO', 'TIME_PERIOD', 'Population active'], how='inner')

display(df_conjugality_final)

## Household

### Suppression des données inutiles

- Suppression des colonnes FREQ et OCS qui comportent une seule valeur
- GEO_OBJECT : on conserve uniquement la valeur 'COM' (commune)
- Supression de GEO_OBJECT après avoir filtré les communes
- RP_MEASURE : on conserve la mesure DWELLINGS_POPSIZE (population des ménages)
- PREFPH : on conserve les valeurs liées à l'ensemble du ménage, pas uniquement aux personnes référentes
- Suppression de la colonne PCS qui n'a qu'une seule valeur après les traitements précédents

In [ ]:
# Suppression des lignes et colonnes inutiles

df_household = df_household[(df_household['GEO_OBJECT'] == 'COM') & (df_household['RP_MEASURE'] == 'DWELLINGS_POPSIZE') & (df_household['PREFPH'] == '_T')]
df_household = df_household.drop(['GEO_OBJECT', 'FREQ', 'OCS', 'RP_MEASURE', 'PREFPH', 'PCS'], axis=1)
df_household = df_household.sort_values(['TIME_PERIOD', 'GEO'])



### Premier aperçu des données

In [ ]:
duplicates_count = df_household.duplicated().sum()
print('Nombre de doublons :', duplicates_count)

null_values_count = df_household.isnull().sum()
print(f'\nNombre de valeurs manquantes : {null_values_count}')

distinct_cities = len(df_household['GEO'].unique())
print(f'\nNombre de communes : {distinct_cities}')

display(df_household)

### Structuration des données

- Transformer la colonne TPH en une colonne par composition de ménage

In [ ]:
df_household_final = df_household.pivot_table(
    index=['GEO', 'TIME_PERIOD'],
    columns='TPH',
    values='OBS_VALUE',
    aggfunc='sum',
    fill_value=0
).rename(columns={
    '11': 'Personne seule',
    '110': 'Homme seul',
    '111': 'Femme seule',
    '12': 'Colocation',
    '2': 'Famille',
    'MF21': 'Famille monoparentale',
    'MF221': 'Couple sans enfant',
    'MF222': 'Couple avec enfants',
    '_T': 'Population avec enfants'
})

df_household_final = df_household_final.reset_index()

display(df_household_final)

## Dataframe INSEE final

In [ ]:
# On vérifie s'il y a des communes manquantes dans l'un des dataframes :
print('Nombre de communes population :', len(df_population_final))
print('Nombre de communes conjugalité :', len(df_conjugality_final))
print('Nombre de communes ménages :', len(df_household_final))
print('---> 15 lignes supplémentaires dans le dataframe conjugalité')


# On fusionne les 3 df en conservant toutes les lignes et on affiche celles qui ont des valeurs manquantes
df_merged = pd.merge(df_population_final, df_household_final, on=['TIME_PERIOD', 'GEO'])
df_insee = pd.merge(df_merged, df_conjugality_final, on=['TIME_PERIOD', 'GEO'], how='outer')
df_insee = df_insee.rename(columns={'GEO': 'Code_INSEE', 'TIME_PERIOD': 'Année'})


# Pour les colonnes Population active_x et Population active_y, on garde la valeur maximale pour chaque commune$
df_insee['Population_active'] = df_insee[['Population active_x', 'Population active_y']].max(axis=1)
df_insee = df_insee.drop(['Population active_x', 'Population active_y'], axis=1)

display(df_insee)


In [ ]:
# On étudie les valeurs manquantes
df_null = df_insee[df_insee.isna().any(axis=1)]
display(df_null.sort_values(['Code_INSEE', 'Année']))

print('Communes ayant des valeurs manquantes (à traiter côté modélisation) :')
display(df_null['Code_INSEE'].unique())

## Export du dataframe

In [ ]:
from pathlib import Path

if Path('src/insee.csv').exists():
    print('Le fichier existe déjà, supprimez-le pour le mettre à jour')
else:
    df_insee.to_csv('src/insee.csv', index=False, sep=';', encoding='utf-8')